# Real DWI Data PCA Demo

This demo demonstrates PCA analysis on real diffusion-weighted imaging
(DWI) data from a single brain slice acquired with multiple gradient
directions.

The analysis treats:
- Voxels as features (spatial locations in the brain slice)
- Gradient directions as samples (different diffusion measurements)

This provides insight into the principal diffusion patterns across
the brain slice.

In [ ]:
import matplotlib.pyplot as pltimport numpy as npfrom dipy.data import fetch_stanford_hardi, read_stanford_hardifrom pipelinecombat.model.pca import PCA

In [ ]:
def fetch_real_dwi_data(normalize_by_b0=True):    """    Fetch real DWI data using DIPY's Stanford HARDI dataset.    Optionally normalize diffusion-weighted signals by b0 (baseline) signal.    Parameters    ----------    normalize_by_b0 : bool, optional        If True, normalize DWI signals by averaged b0 signal and return only        diffusion-weighted volumes. If False, return all volumes including b0.        Default is True.    Returns    -------    dwi_data : ndarray        DWI data from a single slice. If normalize_by_b0=True:        (n_dwi_directions, n_voxels) - b0 normalized, DWI only.        If normalize_by_b0=False: (n_all_directions, n_voxels) - raw data.    gtab : GradientTable        Gradient table. If normalize_by_b0=True: only DWI directions.        If normalize_by_b0=False: all directions including b0.    """    print("🔄 Fetching Stanford HARDI dataset...")    # Fetch the dataset (publicly available, no registration needed)    fetch_stanford_hardi()    img, gtab = read_stanford_hardi()    # Get the DWI data    dwi_data = img.get_fdata()  # Shape: (x, y, z, directions)    print(f"Original DWI shape: {dwi_data.shape}")    # Select a representative axial slice (middle of the brain)    z_slice = dwi_data.shape[2] // 2    dwi_slice_4d = dwi_data[:, :, z_slice, :]  # Shape: (x, y, directions)    # Reshape to (directions, voxels) - treating voxels as features    n_x, n_y, n_directions = dwi_slice_4d.shape    dwi_slice = dwi_slice_4d.reshape(n_x * n_y, n_directions).T    # Remove background voxels (very low signal)    mean_signal = np.mean(dwi_slice, axis=0)    signal_threshold = np.percentile(mean_signal, 10)  # Keep 90% of voxels    valid_voxels = mean_signal > signal_threshold    dwi_slice = dwi_slice[:, valid_voxels]    print(        f"Selected slice {z_slice}, shape before processing: {dwi_slice.shape}"    )    if not normalize_by_b0:        # Return raw data without normalization        print("Using raw DWI data (no b0 normalization)")        print(            f"({dwi_slice.shape[0]} gradient directions, "            f"{dwi_slice.shape[1]} brain voxels)"        )        print(            f"B-values range: {gtab.bvals.min():.0f} - "            f"{gtab.bvals.max():.0f} s/mm²"        )        print(            f"Raw signal range: {dwi_slice.min():.1f} - {dwi_slice.max():.1f}"        )        return dwi_slice, gtab    # Normalize by b0 (original behavior)    print("Applying b0 normalization...")    # Separate b0 and diffusion-weighted volumes    b0_mask = gtab.bvals <= 50  # b0 volumes (allowing small tolerance)    dwi_mask = gtab.bvals > 50  # diffusion-weighted volumes    b0_volumes = dwi_slice[b0_mask, :]    dwi_volumes = dwi_slice[dwi_mask, :]    print(        f"Found {np.sum(b0_mask)} b0 volumes and "        f"{np.sum(dwi_mask)} DWI volumes"    )    # Average b0 volumes if multiple exist    if b0_volumes.shape[0] > 1:        b0_mean = np.mean(b0_volumes, axis=0)        print(f"Averaged {b0_volumes.shape[0]} b0 volumes")    else:        b0_mean = b0_volumes[0, :]        print("Using single b0 volume")    # Normalize DWI signals by b0 (avoid division by zero)    # Add small epsilon to prevent division by very small numbers    epsilon = 1e-6    b0_mean_safe = np.maximum(b0_mean, epsilon)    # Calculate normalized signal: DWI / b0    dwi_normalized = dwi_volumes / b0_mean_safe[np.newaxis, :]    # Create gradient table for only DWI volumes    gtab_dwi = gtab[dwi_mask]    print(f"Final normalized DWI shape: {dwi_normalized.shape}")    print(        f"({dwi_normalized.shape[0]} gradient directions, "        f"{dwi_normalized.shape[1]} brain voxels)"    )    print(        f"B-values range: {gtab_dwi.bvals.min():.0f} - "        f"{gtab_dwi.bvals.max():.0f} s/mm²"    )    print(        f"Normalized signal range: {dwi_normalized.min():.4f} - "        f"{dwi_normalized.max():.4f}"    )    return dwi_normalized, gtab_dwi

In [ ]:
def visualize_comprehensive_dwi_pca(    dwi_data, gtab, pca_model, n_components=20

In [ ]:
def visualize_dwi_loadings_analysis(pca_model, dwi_data, gtab):    """    Visualize PCA component loadings in detail for DWI data.    Parameters    ----------    pca_model : PCA        The fitted PCA model    dwi_data : ndarray        DWI data matrix (n_directions, n_voxels)    gtab : GradientTable        Gradient table information    """    print("Creating detailed DWI loadings analysis...")    # Use the model's public interface    model = pca_model.generate()    components = model.I  # Shape: (n_components+1, n_features)    all_variances = model.variances    # Skip mean component for analysis    pca_components = components[1:7, :]  # First 6 PCA components    pca_variances = all_variances[1:7]  # Corresponding variances    total_var = np.sum(        all_variances[1:]    )  # Total PCA variance (excluding mean)    explained_variance_ratio = pca_variances / total_var    n_comp = min(6, pca_components.shape[0])    n_features_per_component = 10    # Create figure with subplots for each component    fig, axes = plt.subplots(2, 3, figsize=(18, 12))    fig.suptitle(        "🧠 DWI Component Loadings Analysis\nSpatial Diffusion Patterns",        fontsize=16,        fontweight="bold",    )    axes = axes.flatten()    for i in range(n_comp):        ax = axes[i]        component_loadings = pca_components[i, :]  # Get component loadings        # Get top positive and negative loadings        abs_loadings = np.abs(component_loadings)        top_indices = np.argsort(abs_loadings)[-n_features_per_component:][            ::-1        ]        # Create horizontal bar plot        top_loadings = component_loadings[top_indices]        colors = ["red" if x < 0 else "blue" for x in top_loadings]        bars = ax.barh(            range(len(top_loadings)), top_loadings, color=colors, alpha=0.7        )        # Set labels (use voxel indices as feature names)        labels = [f"Voxel {idx}" for idx in top_indices]        ax.set_yticks(range(len(top_loadings)))        ax.set_yticklabels(labels, fontsize=8)        ax.set_xlabel("Loading Value", fontsize=10)        ax.set_title(            f"PC{i + 1} ({explained_variance_ratio[i]:.1%} var)\n"            "Top Voxel Loadings",            fontweight="bold",        )        ax.axvline(x=0, color="black", linestyle="-", alpha=0.5)        ax.grid(visible=True, alpha=0.3)        # Add value labels        for _j, (bar, loading) in enumerate(            zip(bars, top_loadings, strict=False)        ):            ax.text(                loading + (0.001 if loading >= 0 else -0.001),                bar.get_y() + bar.get_height() / 2,                f"{loading:.3f}",                ha="left" if loading >= 0 else "right",                va="center",                fontsize=8,            )    # Hide unused subplots (shouldn't be any for 6 components)    for i in range(n_comp, len(axes)):        axes[i].set_visible(False)    plt.tight_layout()    # Print detailed loading statistics    print("\n" + "=" * 70)    print("DWI COMPONENT LOADINGS ANALYSIS")    print("=" * 70)    for i in range(n_comp):        component_loadings = pca_components[i, :]        variance_pct = explained_variance_ratio[i] * 100        print(f"\nPC{i + 1} ({variance_pct:.1f}% variance):")        print(            f"  Range: [{np.min(component_loadings):.3f}, "            f"{np.max(component_loadings):.3f}]"        )        mean_abs_loading = np.mean(np.abs(component_loadings))        print(f"  Mean absolute loading: {mean_abs_loading:.3f}")        print(f"  Std of loadings: {np.std(component_loadings):.3f}")        # Top positive and negative loadings        pos_loadings = component_loadings[component_loadings > 0]        neg_loadings = component_loadings[component_loadings < 0]        if len(pos_loadings) > 0:            max_pos_idx = np.argmax(component_loadings)            print(                f"  Strongest positive loading: Voxel {max_pos_idx} "                f"({component_loadings[max_pos_idx]:.3f})"            )        if len(neg_loadings) > 0:            max_neg_idx = np.argmin(component_loadings)            print(                f"  Strongest negative loading: Voxel {max_neg_idx} "                f"({component_loadings[max_neg_idx]:.3f})"            )    # Additional DWI-specific analysis    print("\nDWI-Specific Loading Patterns:")    print(f"  Total brain voxels: {pca_components.shape[1]}")    print(f"  Gradient directions: {dwi_data.shape[0]}")    print(        f"  B-value range: {gtab.bvals.min():.0f} - {gtab.bvals.max():.0f} "        "s/mm²"    )    return fig

In [ ]:
def visualize_dwi_data_distribution(dwi_data, gtab, pca_model):    """    Visualize data distribution before/after PCA transformation for DWI data.    Parameters    ----------    dwi_data : ndarray        DWI data matrix (n_directions, n_voxels)    gtab : GradientTable        Gradient table information    pca_model : PCA        The fitted PCA model    """    print("Creating DWI data distribution analysis...")    # Use model's public interface    model = pca_model.generate()    all_variances = model.variances    explained_variance_ratio = all_variances[1:] / np.sum(all_variances[1:])    # Generate transformed data    transformed_data = []    for i in range(dwi_data.shape[0]):        beta, _ = model.fit(dwi_data[i, :])        transformed_data.append(beta)    transformed_data = np.array(transformed_data)    n_components = transformed_data.shape[1] - 1  # Exclude mean component    # Create figure    fig, axes = plt.subplots(2, 3, figsize=(18, 12))    fig.suptitle(        "🧠 DWI Data Distribution Analysis\n"        "Before and After PCA Transformation",        fontsize=16,        fontweight="bold",    )    # 1. Original data distribution (first few voxels)    n_voxels_to_show = min(5, dwi_data.shape[1])    voxel_indices = np.linspace(        0, dwi_data.shape[1] - 1, n_voxels_to_show, dtype=int    )    for _i, voxel_idx in enumerate(voxel_indices):        axes[0, 0].hist(            dwi_data[:, voxel_idx],            bins=20,            alpha=0.6,            label=f"Voxel {voxel_idx}",            density=True,        )    axes[0, 0].set_xlabel("Signal Intensity")    axes[0, 0].set_ylabel("Density")    axes[0, 0].set_title("Original DWI Signal Distribution\n(Sample Voxels)")    axes[0, 0].legend(fontsize=8)    axes[0, 0].grid(visible=True, alpha=0.3)    # 2. Transformed data distribution (first few components, skip mean)    n_comp_to_show = min(5, n_components)    for i in range(n_comp_to_show):        axes[0, 1].hist(            transformed_data[:, i + 1],  # Skip mean component at index 0            bins=20,            alpha=0.6,            label=f"PC{i + 1}",            density=True,        )    axes[0, 1].set_xlabel("Coefficient Value")    axes[0, 1].set_ylabel("Density")    axes[0, 1].set_title("PCA Coefficients Distribution")    axes[0, 1].legend(fontsize=8)    axes[0, 1].grid(visible=True, alpha=0.3)    # 3. PC1 vs PC2 colored by B-value    if n_components >= 2:        b_values = gtab.bvals        scatter = axes[0, 2].scatter(            transformed_data[:, 1],  # PC1 (skip mean at index 0)            transformed_data[:, 2],  # PC2            c=b_values[: len(transformed_data)],            cmap="viridis",            alpha=0.7,            s=40,        )        axes[0, 2].set_xlabel(f"PC1 ({explained_variance_ratio[0]:.1%})")        axes[0, 2].set_ylabel(f"PC2 ({explained_variance_ratio[1]:.1%})")        axes[0, 2].set_title("PC1 vs PC2 Distribution\n(Colored by B-value)")        axes[0, 2].grid(visible=True, alpha=0.3)        plt.colorbar(            scatter, ax=axes[0, 2], shrink=0.8, label="B-value (s/mm²)"        )    # 4. Voxel variance before PCA    voxel_variances = np.var(dwi_data, axis=0)    n_voxels_show = min(50, len(voxel_variances))    voxel_subset = np.linspace(        0, len(voxel_variances) - 1, n_voxels_show, dtype=int    )    axes[1, 0].bar(        range(n_voxels_show),        voxel_variances[voxel_subset],        alpha=0.7,        color="orange",    )    axes[1, 0].set_xlabel("Voxel Index (subset)")    axes[1, 0].set_ylabel("Signal Variance")    axes[1, 0].set_title("Voxel Signal Variances\n(Spatial Variability)")    axes[1, 0].grid(visible=True, alpha=0.3)    # 5. PC coefficient variance    pc_variances = np.var(        transformed_data[:, 1: n_comp_to_show + 1], axis=0    )  # Skip mean    axes[1, 1].bar(        range(len(pc_variances)), pc_variances, alpha=0.7, color="green"    )    axes[1, 1].set_xlabel("Principal Component")    axes[1, 1].set_ylabel("Coefficient Variance")    axes[1, 1].set_title("PC Coefficient Variances")    axes[1, 1].set_xticks(range(len(pc_variances)))    axes[1, 1].set_xticklabels(        [f"PC{i + 1}" for i in range(len(pc_variances))]    )    axes[1, 1].grid(visible=True, alpha=0.3)    # 6. B-value vs Signal Analysis    b_values = gtab.bvals    unique_b_vals = np.unique(b_values)    # Calculate mean signal for each b-value shell    b_val_signals = []    b_val_errors = []    for b_val in unique_b_vals:        b_indices = b_values == b_val        if np.sum(b_indices) > 0:            b_signals = np.mean(                dwi_data[b_indices, :], axis=1            )  # Mean across voxels            b_val_signals.append(np.mean(b_signals))            b_val_errors.append(np.std(b_signals))    axes[1, 2].errorbar(        unique_b_vals,        b_val_signals,        yerr=b_val_errors,        marker="o",        capsize=5,        capthick=2,        linewidth=2,        alpha=0.8,    )    axes[1, 2].set_xlabel("B-value (s/mm²)")    axes[1, 2].set_ylabel("Mean Signal Intensity")    axes[1, 2].set_title("Signal Attenuation vs B-value\n(Diffusion Decay)")    axes[1, 2].grid(visible=True, alpha=0.3)    # Add exponential decay reference line if we have different b-values    if len(unique_b_vals) > 1:        b_range = np.linspace(unique_b_vals.min(), unique_b_vals.max(), 100)        # Simple mono-exponential model: S = S0 * exp(-b * ADC)        if len(b_val_signals) >= 2:            # Estimate ADC from first two points            s0, s1 = b_val_signals[0], b_val_signals[1]            b0, b1 = unique_b_vals[0], unique_b_vals[1]            if s1 > 0 and b1 > b0:                adc_est = -np.log(s1 / s0) / (b1 - b0)                decay_model = s0 * np.exp(-b_range * adc_est)                axes[1, 2].plot(                    b_range,                    decay_model,                    "--",                    alpha=0.5,                    label=f"Mono-exp (ADC≈{adc_est:.3f})",                    color="red",                )                axes[1, 2].legend(fontsize=8)    plt.tight_layout()    # Print distribution statistics    print("\n" + "=" * 60)    print("DWI DATA DISTRIBUTION SUMMARY")    print("=" * 60)    print(f"Original data shape: {dwi_data.shape}")    print(f"Transformed data shape: {transformed_data.shape}")    print("\nOriginal data statistics:")    print(f"  Mean: {np.mean(dwi_data):.4f}")    print(f"  Std: {np.std(dwi_data):.4f}")    print(f"  Min: {np.min(dwi_data):.4f}")    print(f"  Max: {np.max(dwi_data):.4f}")    print("\nTransformed data statistics (PCA coefficients):")    pca_coeffs = transformed_data[:, 1:]  # Exclude mean component    print(f"  Mean: {np.mean(pca_coeffs):.4f}")    print(f"  Std: {np.std(pca_coeffs):.4f}")    print(f"  Min: {np.min(pca_coeffs):.4f}")    print(f"  Max: {np.max(pca_coeffs):.4f}")    print("\nB-value distribution:")    for b_val, count in zip(        *np.unique(b_values, return_counts=True), strict=False    ):        print(f"  {int(b_val):4d} s/mm²: {count:2d} directions")    return fig

In [ ]:
def visualize_dwi_reconstruction_and_distances(    pca_model, dwi_data, n_samples_for_distances=50

In [ ]:
def main():    """Main function to run the real DWI PCA demo."""    print("🧠 Real DWI Data PCA Demo")    print("=" * 50)    # Control parameters    use_b0_normalization = True  # Set to False to use raw DWI data    print(        "📊 B0 normalization: "        f"{'Enabled' if use_b0_normalization else 'Disabled'}"    )    try:        # Fetch real DWI data        dwi_data, gtab = fetch_real_dwi_data(            normalize_by_b0=use_b0_normalization        )        # Apply PCA        print("\n🔄 Applying PCA to DWI data...")        n_components = min(12, dwi_data.shape[0] - 1)  # Don't exceed samples        pca_dwi = PCA(numerical_data=dwi_data, n_components=n_components)        print(f"PCA model created with {len(pca_dwi._n_variance)} components")        # Create visualizations (matching pca_demo structure)        print("\n1. Creating comprehensive DWI PCA analysis...")        visualize_comprehensive_dwi_pca(dwi_data, gtab, pca_dwi, n_components)        print("\n2. Creating detailed component loadings analysis...")        visualize_dwi_loadings_analysis(pca_dwi, dwi_data, gtab)        print("\n3. Creating data distribution analysis...")        visualize_dwi_data_distribution(dwi_data, gtab, pca_dwi)        print(            "\n4. Creating reconstruction and distance preservation plots..."        )        visualize_dwi_reconstruction_and_distances(pca_dwi, dwi_data)        # Show all plots        plt.show()        print("\n🎉 Real DWI PCA Demo completed successfully!")        print("=" * 50)    except Exception as e:        print(f"❌ Error in DWI demo: {e!s}")        raise

In [ ]:
# Main executionprint("🧠 Real DWI Data PCA Demo")print("=" * 50)# Control parametersuse_b0_normalization = True  # Set to False to use raw DWI dataprint("📊 B0 normalization: " f"{'Enabled' if use_b0_normalization else 'Disabled'}")try:    # Fetch real DWI data    dwi_data, gtab = fetch_real_dwi_data(normalize_by_b0=use_b0_normalization)    # Apply PCA    print("\n🔄 Applying PCA to DWI data...")    n_components = min(12, dwi_data.shape[0] - 1)  # Don't exceed samples    pca_dwi = PCA(numerical_data=dwi_data, n_components=n_components)    print(f"PCA model created with {len(pca_dwi._n_variance)} components")    # Create visualizations    print("\n1. Creating comprehensive DWI PCA analysis...")    visualize_comprehensive_dwi_pca(dwi_data, gtab, pca_dwi, n_components)    print("\n2. Creating detailed component loadings analysis...")    visualize_dwi_loadings_analysis(pca_dwi, dwi_data, gtab)    print("\n3. Creating data distribution analysis...")    visualize_dwi_data_distribution(dwi_data, gtab, pca_dwi)    print("\n4. Creating reconstruction and distance preservation plots...")    visualize_dwi_reconstruction_and_distances(pca_dwi, dwi_data)    # Show all plots    plt.show()    print("\n🎉 Real DWI PCA Demo completed successfully!")    print("=" * 50)except Exception as e:    print(f"❌ Error in DWI demo: {e!s}")    raise